In [4]:
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
import torch
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


rng = np.random.default_rng(seed=123)

In [5]:
def setup_device():
    try:
        import torch_directml

        device = torch_directml.device()
        backend = "directml"    
    except ImportError:
        if torch.cuda.is_available():
            device = torch.device("cuda")
            backend = "cuda"
        else:
            device = torch.device("cpu")
            backend = "cpu"
    return device

device = setup_device()

In [6]:
# ---- Define the PyTorch model ----
class SimpleNN(nn.Module):
    def __init__(self, d):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(d * d, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, d),
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x


In [7]:
# --- Przygotowanie danych MNIST (resize do d x d) i DataLoaderów
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import torch

# Parametry (możesz je zmienić)
d = 28  # wymiar wejściowy/wyjściowy; sieć nie zmienia architektury, więc d to liczba klas
batch_size = 256
epochs = 5

transform = transforms.Compose([
    transforms.Resize((d, d)),
    transforms.ToTensor(),
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Konwersja do tensora i DataLoader
X_train = train_ds.data.unsqueeze(1).float() / 255.0  # (N,1,28,28)
X_test = test_ds.data.unsqueeze(1).float() / 255.0

# jeśli transform zmienia rozmiar, użyj transformów ponownie per-próbka
if transform.transforms[0].size != (28, 28):
    # zmiana rozmiaru wymaga ponownej aplikacji transformacji do każdej próbki
    from torchvision.transforms import ToPILImage
    X_train = torch.stack([transform(ToPILImage()(img)) for img in train_ds.data])
    X_test = torch.stack([transform(ToPILImage()(img)) for img in test_ds.data])

y_train = train_ds.targets
y_test = test_ds.targets

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

print(f"Prepared MNIST: train={len(train_ds)}, test={len(test_ds)}, image size={(d,d)}")

Prepared MNIST: train=60000, test=10000, image size=(28, 28)


In [8]:
# --- Trening SimpleNN i ewaluacja na zbiorze testowym
import torch
import torch.nn as nn
import torch.optim as optim

# Zakładamy, że klasa SimpleNN i zmienna `device` są już zdefiniowane w poprzednich komórkach
model = SimpleNN(d).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)

    avg_loss = running_loss / len(train_loader.dataset)

    # oblicz dokładność na zbiorze treningowym i testowym
    def eval_loader(loader):
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        return correct / total

    train_acc = eval_loader(train_loader)
    test_acc = eval_loader(test_loader)

    print(f"Epoch {epoch}/{epochs} - loss: {avg_loss:.4f} - train_acc: {train_acc:.4f} - test_acc: {test_acc:.4f}")

# Final evaluation
final_test_acc = eval_loader(test_loader)
print(f"Final test accuracy: {final_test_acc:.4f}")

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\torch\optim\adam.py:534: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


Epoch 1/5 - loss: 1.5731 - train_acc: 0.8170 - test_acc: 0.8194
Epoch 2/5 - loss: 0.2853 - train_acc: 0.9582 - test_acc: 0.9502
Epoch 2/5 - loss: 0.2853 - train_acc: 0.9582 - test_acc: 0.9502
Epoch 3/5 - loss: 0.1450 - train_acc: 0.9734 - test_acc: 0.9623
Epoch 3/5 - loss: 0.1450 - train_acc: 0.9734 - test_acc: 0.9623
Epoch 4/5 - loss: 0.1014 - train_acc: 0.9794 - test_acc: 0.9696
Epoch 4/5 - loss: 0.1014 - train_acc: 0.9794 - test_acc: 0.9696
Epoch 5/5 - loss: 0.0772 - train_acc: 0.9815 - test_acc: 0.9677
Epoch 5/5 - loss: 0.0772 - train_acc: 0.9815 - test_acc: 0.9677
Final test accuracy: 0.9677
Final test accuracy: 0.9677
